# ML-05 — Feature Vector and Leakage/Privacy Check
## 1. Feature Vector Construction

The goal of this feature vector is to predict refresh opportunities using only information available before the decision moment.

The selected features represent historical user behavior and content engagement signals. Search Console variables used to define the proxy label were intentionally excluded to prevent data leakage.

The final feature set contains:

- ga4_sessions: Number of user sessions reaching the content. It represents observed traffic before the prediction moment.
- ga4_engaged_sessions: Number of sessions with meaningful engagement.
- sessions_organic: Organic traffic received by the content.
- sessions_ai: Traffic generated from AI sources.
- scroll_events: User interaction with the content.
- ga4_total_engagement_sec: Total time users spent engaging with the content.
- ga4_pageviews: Number of content views.

Missing values in GA4-related metrics were replaced with 0 because missing activity represents no observed events.

## 1. Build the feature vector


In [2]:
# ==========================
# IMPORTS
# ==========================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# ==========================
# LOAD DATA
# ==========================

url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

df = pd.read_parquet(url)


print(df.shape)
df.head()


(9841378, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03


## 2. Feature notes (meaning, missing, categorical, available-when?)

## 2. Feature Notes

| Feature | Meaning | Missing Handling | Available When? |
|---|---|---|---|
| ga4_sessions | Measures the number of sessions reaching the content | Filled with 0 | Available before the refresh decision |
| ga4_engaged_sessions | Measures engaged user sessions | Filled with 0 | Available before prediction |
| sessions_organic | Measures organic traffic history | Filled with 0 | Available before prediction |
| sessions_ai | Measures AI referral traffic | Filled with 0 | Available before prediction |
| scroll_events | Measures user interaction with the page | Filled with 0 | Available before prediction |
| ga4_total_engagement_sec | Measures total engagement duration | Filled with 0 | Available before prediction |
| ga4_pageviews | Measures total page views | Filled with 0 | Available before prediction |

All selected features represent historical observations available at the moment when a refresh decision would be made.

In [3]:
# ==========================
# CREATE LABEL
# ==========================


df["ctr"] = (
    df["gsc_clicks"] /
    (df["gsc_impressions"] + 1)
)


df["refresh_opportunity"] = (
    (df["gsc_impressions"] > 50) &
    (df["ctr"] < 0.05) &
    (df["gsc_avg_position"] > 5)
).astype(int)

df["engagement_rate"] = (
    df["ga4_engaged_sessions"] /
    (df["ga4_sessions"] + 1)
)

df["pages_per_session"] = (
    df["ga4_pageviews"] /
    (df["ga4_sessions"] + 1)
)


df["avg_engagement_time"] = (
    df["ga4_total_engagement_sec"] /
    (df["ga4_sessions"] + 1)
)


df["total_sessions"] = (
    df["sessions_organic"] +
    df["sessions_ai"]
)

df["organic_ratio"] = (
    df["sessions_organic"] /
    (df["total_sessions"] + 1)
)

df["ai_ratio"] = (
    df["sessions_ai"] /
    (df["total_sessions"] + 1)
)


df["log_sessions"] = np.log1p(
    df["ga4_sessions"]
)


df["log_pageviews"] = np.log1p(
    df["ga4_pageviews"]
)


df["log_engagement_time"] = np.log1p(
    df["ga4_total_engagement_sec"]
)



# --------------------------
# 8. Low engagement flag
# --------------------------
# Detect content with weak interaction

df["low_engagement"] = (
    df["engagement_rate"] < 0.2
).astype(int)



# --------------------------
# 9. Traffic presence flag
# --------------------------
# Detect pages with any traffic

df["has_traffic"] = (
    df["total_sessions"] > 0
).astype(int)



df["refresh_opportunity"] = (
    (df["gsc_impressions"] > 50) &
    (df["ctr"] < 0.05) &
    (df["gsc_avg_position"] > 5)
).astype(int)

# ==========================
# HANDLE MISSING VALUES
# ==========================


df["gsc_avg_position"] = df["gsc_avg_position"].fillna(
    df["gsc_avg_position"].mean()
)


ga4_features = [
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_ai",
    "scroll_events",
    "ga4_total_engagement_sec",
    "ga4_pageviews"
]


df[ga4_features] = df[ga4_features].fillna(0)


# ==========================
# HONEST FEATURES
# ==========================

features = [

    # Original GA4 signals
    "ga4_sessions",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_ai",
    "scroll_events",
    "ga4_total_engagement_sec",
    "ga4_pageviews",

    # Engineered features
    "engagement_rate",
    "pages_per_session",
    "avg_engagement_time",
    "total_sessions",
    "organic_ratio",
    "ai_ratio",
    "log_sessions",
    "log_pageviews",
    "log_engagement_time",
    "low_engagement",
    "has_traffic"
]


X = df[features].copy()

y = df["refresh_opportunity"]



print("FEATURES USED")
print("----------------")
print(X.columns.tolist())

# ==========================
# SPLIT
# ==========================


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("TRAIN CLASSES")
print("----------------")

print(
    y_train.value_counts()
)

# ==========================
# RANDOM FOREST
# ==========================


rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)


rf_model.fit(
    X_train,
    y_train
)


print("MODEL CLASSES")
print("----------------")

print(
    rf_model.classes_
)

# ==========================
# PREDICTIONS
# ==========================


probs = rf_model.predict_proba(
    X_test
)[:,1]


print(
    "ROC-AUC:"
)

print(
    roc_auc_score(
        y_test,
        probs
    )
)


# ==========================
# THRESHOLD TEST
# ==========================


for threshold in [
    0.3,
    0.4,
    0.5,
    0.6,
    0.7
]:


    pred = (
        probs >= threshold
    ).astype(int)



    print("\nThreshold:", threshold)

    print(
        "Accuracy:",
        accuracy_score(
            y_test,
            pred
        )
    )


    print(
        "Recall:",
        recall_score(
            y_test,
            pred
        )
    )


    print(
        "Precision:",
        precision_score(
            y_test,
            pred,
            zero_division=0
        )
    )


    print(
        "F1:",
        f1_score(
            y_test,
            pred
        )
    )


    # ==========================
# FINAL MODEL
# ==========================


threshold = 0.5


final_pred = (
    probs >= threshold
).astype(int)



print("FINAL RANDOM FOREST MODEL")
print("--------------------------")


print(
    "Accuracy:",
    accuracy_score(
        y_test,
        final_pred
    )
)


print(
    "Recall:",
    recall_score(
        y_test,
        final_pred
    )
)


print(
    "Precision:",
    precision_score(
        y_test,
        final_pred
    )
)


print(
    "F1:",
    f1_score(
        y_test,
        final_pred
    )
)


print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        probs
    )
)


FEATURES USED
----------------
['ga4_sessions', 'ga4_engaged_sessions', 'sessions_organic', 'sessions_ai', 'scroll_events', 'ga4_total_engagement_sec', 'ga4_pageviews', 'engagement_rate', 'pages_per_session', 'avg_engagement_time', 'total_sessions', 'organic_ratio', 'ai_ratio', 'log_sessions', 'log_pageviews', 'log_engagement_time', 'low_engagement', 'has_traffic']
TRAIN CLASSES
----------------
refresh_opportunity
0    7387492
1     485610
Name: count, dtype: int64
MODEL CLASSES
----------------
[0 1]
ROC-AUC:
0.6607627107442022

Threshold: 0.3
Accuracy: 0.061680374093877076
Recall: 1.0
Precision: 0.06167942064976726
F1: 0.11619217524630612

Threshold: 0.4
Accuracy: 0.6597590988255712
Recall: 0.5695787548804797
Precision: 0.10071338893242177
F1: 0.17116180518426108

Threshold: 0.5
Accuracy: 0.9323900713111373
Recall: 0.2932818240226685
Precision: 0.4295814582001134
F1: 0.3485816383973371

Threshold: 0.6
Accuracy: 0.9327380916091036
Recall: 0.2928534950000824
Precision: 0.4330767169342

## 3. The leakage hunt
## 3. Leakage Hunt

A leakage experiment was performed by intentionally adding label-derived features to the model.

The proxy label was created using:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ctr

These variables were tested as model features to demonstrate how including information derived from the target artificially improves performance.

The leaky features were:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ctr

Including these variables causes the model to learn the proxy definition instead of learning generalizable patterns.

The resulting performance increase is considered invalid because these signals would not represent an independent prediction scenario.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score
)


# ==================================
# LEAKY FEATURES
# ==================================

leaky_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr"
]


X_leaky = df[leaky_features].copy()

y = df["refresh_opportunity"]



# ==================================
# SPLIT
# ==================================

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



print("LEAKY FEATURES")
print("----------------")
print(X_leaky.columns.tolist())



# ==================================
# MODEL
# ==================================

leaky_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)


leaky_model.fit(
    X_train,
    y_train
)



# ==================================
# PREDICTIONS
# ==================================

leaky_probs = leaky_model.predict_proba(
    X_test
)[:,1]


leaky_pred = (
    leaky_probs >= 0.5
).astype(int)



# ==================================
# RESULTS
# ==================================

print("\nLEAKY MODEL")
print("----------------")


print(
    "Accuracy:",
    accuracy_score(
        y_test,
        leaky_pred
    )
)


print(
    "Recall:",
    recall_score(
        y_test,
        leaky_pred
    )
)


print(
    "Precision:",
    precision_score(
        y_test,
        leaky_pred
    )
)


print(
    "F1:",
    f1_score(
        y_test,
        leaky_pred
    )
)


print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        leaky_probs
    )
)

LEAKY FEATURES
----------------
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ctr']

LEAKY MODEL
----------------
Accuracy: 1.0
Recall: 1.0
Precision: 1.0
F1: 1.0
ROC-AUC: 1.0


## 4. Excluded Features

The following fields were intentionally excluded from the model:

### gsc_impressions

Excluded because it directly contributes to the refresh opportunity definition. Including it would allow the model to partially reconstruct the target.

### gsc_clicks

Excluded because click behavior is used to calculate the CTR-based proxy label.

### gsc_avg_position

Excluded because ranking position is part of the proxy definition and would create target leakage.

### ctr

Excluded because it is derived from clicks and impressions, which are directly related to the label.

### client_hash_id

Excluded because it is only an identifier and does not represent a meaningful performance signal. Using it could cause the model to memorize specific clients.

### content_hash_id

Excluded because it identifies individual content items rather than representing a generalizable feature.

### Future performance information

Excluded because future observations would not be available at the decision moment and would create unrealistic evaluation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.